# 10 — SLM Attribution Pipeline

Maps flagged event sequences to MITRE ATT\&CK techniques using a local SLM (Phi-4 14B via Ollama).

**Two-model pipeline per flagged chain:**
1. **Sequence AE** flags the chain as anomalous (context-level detection)
2. **Single-event AE** scores every event in the chain individually — the highest-scoring
   events are the ones that deviate most from benign behaviour and are most likely to
   represent the core attack technique
3. **RAG** retrieves the top-*k* candidate ATT\&CK techniques from ChromaDB
4. **Phi-4** reasons over the selected events + candidates and returns a structured attribution

**Prerequisites:** Ollama running locally with `phi4:14b-q4_K_M` pulled.  \
Run `ollama serve` + `ollama pull phi4:14b-q4_K_M` before executing this notebook.

## Imports

In [1]:
import json, pickle, re, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import requests
from pathlib import Path
from collections import Counter
import sys
sys.path.insert(0, '..')

from config import OLLAMA_BASE_URL, OLLAMA_MODEL, OLLAMA_TIMEOUT, RAG_TOP_K, CONTEXT_WINDOW_EVENTS
from data.attack_kb.vector_store import retrieve_hybrid

CKPT_DIR       = Path('checkpoints')
AE_DIR         = CKPT_DIR / 'models' / 'autoencoder'
SEQ_MODEL_DIR  = CKPT_DIR / 'models' / 'seq'
SEQ_CHAINS_DIR = CKPT_DIR / 'seq'
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEQ_THRESHOLD  = 0.627528   # seq AE threshold @ 0.1% FPR (from nb07)
TOP_N_EVENTS   = CONTEXT_WINDOW_EVENTS   # events selected per chain for the SLM (=32)
N_EVAL         = 20         # chains per source for accuracy evaluation

SOURCE_BOUNDS = {
    'otrf_at': (511_105,   1_054_182),
    'splunk':  (1_382_201, 3_447_667),
}

print(f'Device          : {DEVICE}')
print(f'Ollama model    : {OLLAMA_MODEL}')
print(f'Seq threshold   : {SEQ_THRESHOLD}')
print(f'Events per call : {TOP_N_EVENTS}')

Device          : cuda
Ollama model    : qwen2.5:32b
Seq threshold   : 0.627528
Events per call : 32


## 1. Load data and models

Load the malicious event index, both feature matrices, and both AE checkpoints.

| Model | Feature matrix | Role |
|---|---|---|
| Single-event AE (word2vec) | `X_m_w2v.npy` (353 dims) | Per-event anomaly score within a flagged chain |
| Sequence AE (TransformerAE) | `X_m_w2v_norule.npy` (352 dims) | Chain-level anomaly detection |


In [2]:
# ── Events index ─────────────────────────────────────────────────────────────
events_m = pd.read_parquet(CKPT_DIR / 'events_m.parquet')
print(f'events_m : {len(events_m):,} rows')

# ── Feature matrices ──────────────────────────────────────────────────────────
X_m_ae  = np.load(CKPT_DIR / 'word2vec' / 'X_m_w2v.npy',        mmap_mode='r')  # full (AE)
X_m_seq = np.load(CKPT_DIR / 'word2vec' / 'X_m_w2v_norule.npy', mmap_mode='r')  # norule (seq)
print(f'X_m_ae  (single-event AE) : {X_m_ae.shape}')
print(f'X_m_seq (sequence AE)     : {X_m_seq.shape}')

# ── Model definitions ─────────────────────────────────────────────────────────
class Autoencoder(nn.Module):
    _ACT = {'relu': nn.ReLU, 'gelu': nn.GELU,
            'leaky_relu': lambda: nn.LeakyReLU(0.1),
            'selu': nn.SELU, 'elu': nn.ELU}

    def __init__(self, input_dim, layer_sizes, latent_dim,
                 activation='relu', use_batchnorm=True, dropout=0.1):
        super().__init__()
        act_fn = self._ACT[activation]

        def _block(in_d, out_d, final=False):
            layers = [nn.Linear(in_d, out_d)]
            if use_batchnorm and not final:
                layers.append(nn.BatchNorm1d(out_d))
            if not final:
                layers.append(act_fn())
                if dropout > 0:
                    layers.append(nn.Dropout(dropout))
            return layers

        enc_dims = [input_dim] + list(layer_sizes) + [latent_dim]
        enc = []
        for i in range(len(enc_dims) - 1):
            enc.extend(_block(enc_dims[i], enc_dims[i+1], final=(i == len(enc_dims)-2)))
        enc.append(act_fn())
        self.encoder = nn.Sequential(*enc)

        dec_dims = [latent_dim] + list(reversed(layer_sizes)) + [input_dim]
        dec = []
        for i in range(len(dec_dims) - 1):
            dec.extend(_block(dec_dims[i], dec_dims[i+1], final=(i == len(dec_dims)-2)))
        self.decoder = nn.Sequential(*dec)

    def forward(self, x):
        return self.decoder(self.encoder(x))


class TransformerAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, n_layers, nhead, dropout):
        super().__init__()
        if hidden_dim % nhead != 0:
            hidden_dim = max(nhead, (hidden_dim // nhead) * nhead)
        self.input_proj  = nn.Linear(input_dim, hidden_dim)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dim_feedforward=hidden_dim * 2,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.latent_fc   = nn.Linear(hidden_dim, latent_dim)
        self.decode_fc   = nn.Linear(latent_dim, hidden_dim)
        self.output_proj = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        h = self.transformer(self.input_proj(x))
        z = self.latent_fc(h.mean(dim=1))
        d = self.decode_fc(z).unsqueeze(1).expand(-1, x.size(1), -1)
        return self.output_proj(d)

# ── Load single-event AE ─────────────────────────────────────────────────────
ae_best = json.loads((AE_DIR / 'ae_best.json').read_text())['word2vec']
input_dim = X_m_ae.shape[1]
layer_sizes = sorted(
    [max(16, int(input_dim * ae_best[f'ratio_{i}'])) for i in range(ae_best['n_layers'])],
    reverse=True)
latent_dim = max(8, int(input_dim * ae_best['latent_ratio']))
ae_model = Autoencoder(input_dim, layer_sizes, latent_dim,
                       ae_best['activation'], ae_best['use_batchnorm'],
                       ae_best['dropout']).to(DEVICE)
ae_ckpt = torch.load(AE_DIR / 'word2vec' / 'best.pt', map_location=DEVICE, weights_only=True)
ae_model.load_state_dict(ae_ckpt['model_state'])
ae_model.eval()
print(f'Single-event AE : input={input_dim}  layers={layer_sizes}  latent={latent_dim}  '
      f'epoch={ae_ckpt["epoch"]}')

# ── Load sequence AE ─────────────────────────────────────────────────────────
seq_best_raw = json.loads((SEQ_MODEL_DIR / 'best_params.json').read_text())
seq_params   = seq_best_raw.get('params', seq_best_raw)
seq_W        = seq_params['window_size']
seq_stride   = max(1, int(seq_W * seq_params['stride_ratio']))
seq_model    = TransformerAE(X_m_seq.shape[1], seq_params['hidden_dim'],
                             seq_params['latent_dim'], seq_params['n_layers'],
                             seq_params.get('nhead', 4), seq_params['dropout']).to(DEVICE)
seq_state    = torch.load(SEQ_MODEL_DIR / f"seq_ae_{seq_params['strategy']}.pt",
                          map_location=DEVICE, weights_only=True)
seq_model.load_state_dict(seq_state)
seq_model.eval()
print(f'Sequence AE     : arch=transformer  W={seq_W}  stride={seq_stride}  '
      f'input={X_m_seq.shape[1]}  hidden={seq_params["hidden_dim"]}')

events_m : 3,447,667 rows
X_m_ae  (single-event AE) : (3447667, 353)
X_m_seq (sequence AE)     : (3447667, 352)
Single-event AE : input=353  layers=[231, 176, 176, 162]  latent=8  epoch=72
Sequence AE     : arch=transformer  W=45  stride=45  input=352  hidden=256


## 2. Flag anomalous chains

Use the sequence AE to flag chains whose peak window reconstruction error exceeds the
training threshold. One entry is kept per chain (the highest-scoring window) — this is the
dense region of anomalous activity that the single-event AE will later decompose.

In [3]:
with open(SEQ_CHAINS_DIR / 'seq_chains_m.pkl', 'rb') as f:
    chains_m = pickle.load(f)

print(f'Total malicious chains: {len(chains_m):,}')


def filter_chains_by_source(chains, lo, hi):
    return [c for c in chains
            if len(c) > 0 and int(c.min()) >= lo and int(c.max()) < hi]


def flag_chains(model, chains, X, W, stride, threshold, device, batch_size=512):
    """
    For each chain, find the highest-scoring window.
    Returns a list of dicts: {chain, peak_score, peak_start} for chains above threshold.
    """
    model.eval()
    buf, chain_info, all_results = [], [], []

    for ci, chain in enumerate(chains):
        if len(chain) < W:
            continue
        for start in range(0, len(chain) - W + 1, stride):
            buf.append(np.array(X[list(chain[start:start + W])], dtype=np.float32))
            chain_info.append((ci, start))
            if len(buf) >= batch_size:
                x = torch.from_numpy(np.stack(buf)).to(device)
                with torch.no_grad():
                    rec = model(x)
                scores = ((x - rec) ** 2).mean(dim=(1, 2)).cpu().numpy()
                for (ci2, st), sc in zip(chain_info, scores):
                    all_results.append((ci2, st, float(sc)))
                buf, chain_info = [], []

    if buf:
        x = torch.from_numpy(np.stack(buf)).to(device)
        with torch.no_grad():
            rec = model(x)
        scores = ((x - rec) ** 2).mean(dim=(1, 2)).cpu().numpy()
        for (ci2, st), sc in zip(chain_info, scores):
            all_results.append((ci2, st, float(sc)))

    # Keep peak window per chain
    best_per_chain = {}
    for ci, start, sc in all_results:
        if ci not in best_per_chain or sc > best_per_chain[ci][1]:
            best_per_chain[ci] = (start, sc)

    return [
        {'chain': chains[ci], 'peak_score': sc, 'peak_start': start}
        for ci, (start, sc) in best_per_chain.items()
        if sc >= threshold
    ]


flagged = {}

for source, (lo, hi) in SOURCE_BOUNDS.items():
    src_chains = filter_chains_by_source(chains_m, lo, hi)
    print(f'{source}: {len(src_chains)} chains -> scoring...', end=' ', flush=True)
    result = flag_chains(seq_model, src_chains, X_m_seq, seq_W, seq_stride,
                         SEQ_THRESHOLD, DEVICE)
    flagged[source] = result
    print(f'{len(result)} flagged chains')

Total malicious chains: 20,326
otrf_at: 3160 chains -> scoring... 963 flagged chains
splunk: 14590 chains -> scoring... 2351 flagged chains


## 3. Attribution utilities

`attribute_chain` uses deliberately different event sets for RAG and the SLM:

| Step | Events used | Reason |
|---|---|---|
| RAG query | **Full chain** (all events) | Captures every phase — setup, core technique, cleanup. Technique-defining events (e.g. DCOM process creation) may not be the most statistically anomalous. |
| SLM prompt | **AE top-N** (most anomalous) | Keeps the prompt focused on the highest-signal events without noise. |

This decoupling is key: a T1021.003 (DCOM) chain may have Azure registry setup events that the single-event AE scores as most anomalous, while the actual DCOM execution events are only visible in the full chain — so RAG must see the full chain to retrieve the right technique.

In [4]:
_SKIP = {'', '-1', 'nan', 'None', '0', '-', 'N/A', 'n/a'}

RAG_K = 15   # larger k to improve retrieval recall (GT missed at k=5 for ~98% of chains)

# Sysmon EID → behavioural label (used to build a semantic retrieval query)
EID_NAMES = {
    1: 'Process Create', 3: 'Network Connection', 5: 'Process Terminate',
    7: 'Image Load', 8: 'CreateRemoteThread', 10: 'Process Access',
    11: 'File Create', 12: 'Registry Object Create/Delete',
    13: 'Registry Value Set', 15: 'File Create Stream',
    17: 'Pipe Created', 18: 'Pipe Connected', 22: 'DNS Query',
    23: 'File Delete', 25: 'Process Tampering', 26: 'File Delete Detected',
}

# Load KB entries once for Sigma-guided event selection
from data.attack_kb.builder import build_kb
kb_entries = build_kb(force=False)

@torch.no_grad()
def score_events(model, rows, X, device, batch_size=1024):
    """Per-event reconstruction MSE for the given row indices."""
    model.eval()
    all_scores = []
    for start in range(0, len(rows), batch_size):
        batch_rows = rows[start:start + batch_size]
        x = torch.from_numpy(np.array(X[batch_rows], dtype=np.float32)).to(device)
        rec = model(x)
        all_scores.append(((x - rec) ** 2).mean(dim=1).cpu().numpy())
    return np.concatenate(all_scores)

def select_peak_events(chain, ae_model, X_ae, device, top_n=TOP_N_EVENTS, df=None):
    """
    Score every event in the chain with the single-event AE.

    When df is provided, uses stratified selection:
      - Top ceil(top_n/2) events by global AE score
      - Remaining budget: top-2 per EID type not already selected
    This prevents one dominant EID (e.g. registry ops) from crowding out
    technique-defining events of other types (network, process creation, etc.).
    """
    from collections import defaultdict
    rows   = list(chain)
    scores = score_events(ae_model, rows, X_ae, device)

    if df is None:
        top_idx  = np.argsort(scores)[-top_n:]
        top_rows = [rows[i] for i in sorted(top_idx)]
        return top_rows, scores

    # --- global top half ---
    n_global   = (top_n + 1) // 2
    sorted_idx = np.argsort(scores)[::-1]
    global_set = set(int(i) for i in sorted_idx[:n_global])

    # --- stratified top-2 per EID for the remaining budget ---
    remaining  = top_n - len(global_set)
    eid_groups: dict = defaultdict(list)
    for i, r in enumerate(rows):
        if i in global_set:
            continue
        ev  = df.iloc[r]
        eid = ev.get('event_id', '')
        if str(eid).isdigit():
            eid_groups[int(eid)].append((i, float(scores[i])))

    stratified: set = set()
    top_per_eid = max(1, remaining // max(len(eid_groups), 1))
    for eid_rows in eid_groups.values():
        for idx, _ in sorted(eid_rows, key=lambda x: -x[1])[:top_per_eid]:
            stratified.add(idx)
            if len(stratified) >= remaining:
                break
        if len(stratified) >= remaining:
            break

    all_idx  = sorted(global_set | stratified)[:top_n]
    top_rows = [rows[i] for i in sorted(all_idx)]
    return top_rows, scores

def format_events_for_slm(rows: list, df: pd.DataFrame) -> str:
    """Format Sysmon event rows as structured text for the SLM."""
    lines = []
    for i, r in enumerate(rows):
        ev = df.iloc[r]
        eid = ev.get('event_id', '')
        eid_label = EID_NAMES.get(int(eid), '') if str(eid).isdigit() else ''
        parts = [f'Event {i+1}: EID={eid}' + (f' ({eid_label})' if eid_label else '')]
        for field in ['image', 'command_line', 'parent_image', 'parent_cmdline',
                      'image_loaded',
                      'target_object', 'details', 'granted_access',
                      'target_image',
                      'dest_ip', 'dest_hostname', 'target_filename', 'query_name']:
            val = str(ev.get(field, '')).strip()
            if val and val not in _SKIP:
                parts.append(f'  {field}: {val}')
        lines.append('\n'.join(parts))
    return '\n---\n'.join(lines)

def build_retrieval_query(rows: list, df: pd.DataFrame) -> str:
    """
    Build a behavioural summary query for RAG retrieval.
    Maps EID numbers to human-readable labels and summarises key processes/targets,
    bridging the semantic gap between raw Sysmon fields and ATT&CK descriptions.
    """
    eid_counts = Counter()
    images, targets, cmdlines, dest_ips, query_names = [], [], [], [], []

    for r in rows:
        ev = df.iloc[r]
        eid = ev.get('event_id', '')
        if str(eid).isdigit():
            eid_counts[int(eid)] += 1
        for lst, field in [(images,      'image'),
                           (targets,     'target_object'),
                           (targets,     'target_image'),
                           (targets,     'target_filename'),
                           (cmdlines,    'command_line'),
                           (dest_ips,    'dest_ip'),
                           (query_names, 'query_name')]:
            val = str(ev.get(field, '')).strip()
            if val and val not in _SKIP:
                lst.append(val)

    eid_summary = ', '.join(
        f'{EID_NAMES.get(eid, f"EID={eid}")} x{cnt}'
        for eid, cnt in sorted(eid_counts.items(), key=lambda x: -x[1])
    )
    parts = [f'Suspicious Windows Sysmon activity — event types: {eid_summary}.']

    def _top_unique(lst, n=4):
        seen, out = set(), []
        for v in lst:
            key = v.split('\\')[-1].lower()
            if key not in seen:
                seen.add(key)
                out.append(v)
            if len(out) >= n:
                break
        return out

    if images:
        parts.append('Key processes: ' + ', '.join(_top_unique(images)))
    if cmdlines:
        parts.append('Command lines: ' + '; '.join(_top_unique(cmdlines, 2)))
    if targets:
        parts.append('Targets: ' + ', '.join(_top_unique(targets)))
    if dest_ips:
        parts.append('Network destinations: ' + ', '.join(set(dest_ips)))
    if query_names:
        parts.append('DNS queries: ' + ', '.join(set(query_names)))

    return ' '.join(parts)

def get_chain_gt(chain, df):
    """Most common ground-truth ATT&CK technique across the whole chain."""
    rows = list(chain)
    techs = [str(df.iloc[r].get('attck_technique', '')).strip() for r in rows if r < len(df)]
    techs = [t for t in techs if t and t not in _SKIP]
    return Counter(techs).most_common(1)[0][0] if techs else ''


ATTRIBUTION_PROMPT = (
    "You are a cybersecurity analyst reviewing Windows Sysmon events flagged as anomalous.\n\n"
    "## Events in Flagged Chain Window\n"
    "{event_text}\n\n"
    "## Candidate ATT&CK Techniques\n"
    "{candidates_text}\n\n"
    "## Task\n"
    "Work through these steps before giving your final answer:\n\n"
    "**Step 1** - What specific suspicious behaviors do these events show?\n"
    "Focus on: process names, registry paths, command lines, network destinations, EID types.\n\n"
    "**Step 2** - Which candidate technique from the list above best matches these behaviors,\n"
    "and why does it fit better than the other candidates?\n\n"
    "**Step 3** - Return your final answer as valid JSON (no markdown fences).\n"
    "Fields: technique_id, technique_name, confidence (high/medium/low), explanation, recommended_action.\n\n"
)

def ollama_chat(prompt: str) -> str:
    resp = requests.post(
        f'{OLLAMA_BASE_URL}/api/chat',
        json={'model': OLLAMA_MODEL,
              'messages': [{'role': 'user', 'content': prompt}],
              'stream': False,
              'options': {'temperature': 0.1}},
        timeout=OLLAMA_TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()['message']['content']

def parse_json_response(text: str) -> dict:
    import ast as _ast
    # Strip markdown code fences
    clean = re.sub(r'```(?:json)?\s*|```', '', text).strip()
    # Take the LAST complete {...} block (ignores CoT fragments)
    last_open  = clean.rfind("{")
    last_close = clean.rfind("}")
    if last_open != -1 and last_close > last_open:
        blob = clean[last_open:last_close + 1]
        try:
            return json.loads(blob)
        except json.JSONDecodeError:
            try:
                r = _ast.literal_eval(blob)
                if isinstance(r, dict):
                    return {str(k): v for k, v in r.items()}
            except Exception:
                pass
    # Truncated response: no closing '}' — try extracting key fields with regex
    if last_open != -1:
        fragment = clean[last_open:]
        result: dict = {}
        for key in ('technique_id', 'confidence', 'explanation'):
            m = re.search('"' + key + r'"\s*:\s*"([^"]+)"', fragment)
            if m:
                result[key] = m.group(1)
        if 'technique_id' in result:
            return result
    raise ValueError(f'No JSON found in LLM response: {text[:300]}')

def build_multi_queries(rows: list, df, max_queries: int = 3) -> list:
    """
    Build one behavioural query per dominant EID group (up to max_queries),
    plus the combined query as a catch-all.  Multi-query RAG improves recall
    because process-creation, registry, and network events each pull different
    KB entries.
    """
    from collections import defaultdict
    eid_groups: dict = defaultdict(list)
    for r in rows:
        ev  = df.iloc[r]
        eid = ev.get('event_id', '')
        if str(eid).isdigit():
            eid_groups[int(eid)].append(r)
    top_eids = sorted(eid_groups, key=lambda e: -len(eid_groups[e]))[:max_queries]
    queries  = [build_retrieval_query(eid_groups[e], df) for e in top_eids]
    queries.append(build_retrieval_query(rows, df))   # combined fallback
    return list(dict.fromkeys(queries))               # deduplicate, preserve order


def retrieve_multi_query(rows: list, df, k: int = RAG_K) -> list:
    """
    Multi-query RAG: retrieve from each per-EID query + combined query.
    Deduplicates by technique_id, keeping the hit with the best rank
    across all queries.  Returns up to k*2 unique candidates.
    """
    best_hit:  dict = {}
    best_rank: dict = {}
    for query in build_multi_queries(rows, df):
        for rank, hit in enumerate(retrieve_hybrid(query, k=k)):
            tid = hit['technique_id']
            if tid not in best_rank or rank < best_rank[tid]:
                best_rank[tid] = rank
                best_hit[tid]  = hit
    return sorted(best_hit.values(),
                  key=lambda h: best_rank[h['technique_id']])[:k * 2]


def extract_reasoning(raw: str) -> str:
    """Extract the Step 1 behavioural analysis from the model CoT response."""
    # Match **Step 1** (bold) or ### Step 1 / ### Step-by-Step (heading) formats
    m = re.search(
        r'(?:\*{2}|#{2,3}\s*)Step(?:\s*1|\s*-by-Step).*?(?=(?:\*{2}|#{2,3}\s*)Step\s*2|\Z)',
        raw, re.DOTALL | re.IGNORECASE)
    if m:
        lines = [ln for ln in m.group().splitlines()
                 if not re.match(r'^(?:\*{2}|#{2,3})\s*Step', ln.strip(), re.IGNORECASE)]
        text = ''.join(lines).strip()
        if text:
            return text
    # Fallback: first two non-trivial paragraphs before the last JSON block
    cutoff = min((i for i in [raw.rfind('```'), raw.rfind('{')] if i > 0), default=len(raw))
    pre_json = ''.join(
        ln for ln in raw[:cutoff].splitlines()
        if not ln.strip().startswith('```')
    ).strip()
    paras = [p.strip() for p in pre_json.split('\n')
             if len(p.strip()) > 20 and not re.match(r'^#{2,3}\s*Step', p.strip(), re.IGNORECASE)]
    return ''.join(paras[:2]) if paras else ''

def get_sigma_patterns_for_techniques(technique_ids, entries):
    """
    Parse Sigma KB entries for the given technique IDs.
    Returns: dict[technique_id -> {'eids': set[int], 'indicators': list[str]}]

    Sigma entries have description lines like:
      "Sysmon EventID: 10 (category: process_access)"
      "Detection indicators: Image=lsass.exe, GrantedAccess=0x1410"
    """
    patterns = {}
    for tid in technique_ids:
        patterns[tid] = {'eids': set(), 'indicators': []}

    for entry in entries:
        if entry.source != 'Sigma':
            continue
        tid = entry.technique_id
        if tid not in patterns:
            continue
        desc = entry.description
        for line in desc.splitlines():
            line = line.strip()
            if line.startswith('Sysmon EventID:'):
                # e.g. "Sysmon EventID: 10 (category: process_access)"
                parts = line.split(':', 1)[1].strip().split()[0]
                try:
                    patterns[tid]['eids'].add(int(parts))
                except ValueError:
                    pass
            elif line.startswith('Detection indicators:'):
                # e.g. "Detection indicators: Image=lsass, GrantedAccess=0x1410"
                raw_vals = line.split(':', 1)[1].strip()
                for item in raw_vals.split(','):
                    item = item.strip()
                    # keep the value part after '=' (lowercase, stripped)
                    val = item.split('=', 1)[-1].strip().lower()
                    if val and val not in patterns[tid]['indicators']:
                        patterns[tid]['indicators'].append(val)
    return patterns

def select_events_sigma_guided(chain, df, sigma_patterns, ae_model, X_ae, device,
                                top_n=TOP_N_EVENTS, top1_tid=None):
    """
    EID-stratified event selection with Sigma-priority boost.

    Replaces AE-score-based selection, which was dominated by high-volume EID
    types (e.g. 38 registry events) that scored high simply because their
    specific paths were statistically rare — not because they define the technique.

    Algorithm:
      1. Group chain events by EID type.
      2. Priority EIDs (top1_tid's Sigma EIDs present in chain) get 50% of budget.
      3. Remaining budget split evenly across other EID types (min 2 per type).
      4. Within each group, rank by Sigma indicator match count.
      5. Return in chronological (row-index) order.
    """
    from collections import defaultdict

    rows = list(chain)
    if not rows:
        return [], np.array([])

    # ── Group by EID ───────────────────────────────────────────────────────
    eid_groups: dict = defaultdict(list)
    for r in rows:
        ev  = df.iloc[r]
        eid = ev.get('event_id', 0)
        try:
            eid = int(eid)
        except (ValueError, TypeError):
            eid = 0
        eid_groups[eid].append(r)

    # ── Sigma indicator set for intra-group ranking ────────────────────────
    seen_ind: set = set()
    indicators: list = []
    for pat in list(sigma_patterns.values())[:3]:
        for ind in pat.get('indicators', []):
            if ind not in seen_ind:
                seen_ind.add(ind)
                indicators.append(ind)

    def indicator_score(r: int) -> int:
        ev  = df.iloc[r]
        s   = ' '.join(str(v).lower() for v in ev if isinstance(v, str) and v)
        return sum(1 for ind in indicators if ind in s)

    # Rank each EID group: indicator hits first, then chronological
    ranked: dict = {}
    for eid, grp in eid_groups.items():
        ranked[eid] = sorted(grp, key=indicator_score, reverse=True) if indicators else grp

    # ── Budget allocation ──────────────────────────────────────────────────
    priority_eids: set = set()
    if top1_tid and top1_tid in sigma_patterns:
        priority_eids = sigma_patterns[top1_tid]['eids']
    elif sigma_patterns:
        priority_eids = next(iter(sigma_patterns.values()))['eids']

    priority_present = [e for e in priority_eids if e in eid_groups]
    other_eids       = [e for e in eid_groups   if e not in priority_eids]

    allocation: dict = {}

    if priority_present:
        # Priority EIDs share 50% of budget
        priority_budget = max(top_n // 2, len(priority_present))
        per_priority    = max(1, priority_budget // len(priority_present))
        for eid in priority_present:
            allocation[eid] = min(per_priority, len(eid_groups[eid]))

    remaining = top_n - sum(allocation.values())

    if other_eids and remaining > 0:
        base = max(2, remaining // len(other_eids))
        for eid in other_eids:
            allocation[eid] = min(base, len(eid_groups[eid]))

    # Distribute leftover slots to largest groups first
    surplus = top_n - sum(allocation.values())
    if surplus > 0:
        for eid in sorted(eid_groups, key=lambda e: -len(eid_groups[e])):
            if surplus <= 0:
                break
            headroom = len(eid_groups[eid]) - allocation.get(eid, 0)
            give     = min(headroom, surplus)
            allocation[eid] = allocation.get(eid, 0) + give
            surplus -= give

    # ── Select and return in chronological order ───────────────────────────
    selected = []
    for eid, n_sel in allocation.items():
        selected.extend(ranked[eid][:n_sel])

    return sorted(set(selected)), np.zeros(len(rows))

def attribute_chain(entry: dict, events_m: pd.DataFrame, k: int = RAG_K) -> dict:
    """
    Two-pass attribution pipeline:
      Pass 1 (RAG): multi-query retrieval from full chain to find candidate techniques
      Pass 2 (Sigma): use Sigma patterns of top candidates to weight event selection
      Pass 3 (LLM): attribute using Sigma-guided events as context

    This corrects the single-event AE flaw: AE selects statistically rare events
    (registry writes, image loads) rather than technique-defining ones.  Sigma
    patterns tell us which EID types and field values are technique-relevant.
    """
    chain    = entry['chain']
    all_rows = list(chain)

    # Pass 1: multi-query RAG from full chain to get candidate techniques
    query_rows = all_rows if len(all_rows) <= 300 else sorted(random.sample(all_rows, 300))
    hits       = retrieve_multi_query(query_rows, events_m, k=k)

    # Top-5 technique IDs for Sigma pattern lookup; top-1 drives EID selection
    top_tids  = list(dict.fromkeys(h['technique_id'] for h in hits[:5]))
    top1_tid  = top_tids[0] if top_tids else None
    sigma_patterns = get_sigma_patterns_for_techniques(top_tids, kb_entries)

    # Pass 2: send the exact anomalous window to the LLM.
    # flag_chains stores the full chain; peak_start marks the highest-scoring
    # seq_W-event window within it.  Slice that window — no further selection.
    peak_start   = entry.get('peak_start', 0)
    peak_rows    = all_rows[peak_start : peak_start + seq_W]
    if not peak_rows:           # peak_start out of bounds guard
        peak_rows = all_rows[:seq_W]
    event_scores = np.zeros(len(all_rows))

    event_text = format_events_for_slm(peak_rows, events_m)

    # Build event-type summary and selected EID set
    eid_counts: Counter = Counter()
    for r in peak_rows:
        ev  = events_m.iloc[r]
        eid = ev.get('event_id', '')
        if str(eid).isdigit():
            eid_counts[int(eid)] += 1
    event_summary = ', '.join(
        f'EID={e}({EID_NAMES.get(e, "?")}) x{n}'
        for e, n in sorted(eid_counts.items(), key=lambda x: -x[1])
    )
    sigma_eids_used = sorted(
        sigma_patterns[top1_tid]['eids'] if top1_tid and top1_tid in sigma_patterns
        else {eid for pat in sigma_patterns.values() for eid in pat['eids']}
    )

    # All RAG candidates in original retrieval order.
    # Re-ranking by Sigma-EID overlap was removed: it promoted generic registry
    # techniques (T1562) above specific ones (T1003) for registry-heavy chains.
    candidates_text = ''.join(f'[{h["technique_id"]}] {h["technique_name"]}: {h["document"]}'
        for h in hits
    )
    valid_ids        = ', '.join(dict.fromkeys(h['technique_id'] for h in hits))
    candidates_text += f'Valid technique IDs (you MUST choose one of these): {valid_ids}'

    prompt = ATTRIBUTION_PROMPT.format(
        event_text=event_text,
        candidates_text=candidates_text,
    )
    t0  = time.time()
    raw = ollama_chat(prompt)
    elapsed = time.time() - t0

    result = parse_json_response(raw)
    result['technique_id']      = str(result.get('technique_id') or '').strip()
    result['peak_rows']         = peak_rows
    result['rag_candidates']    = [h['technique_id'] for h in hits]  # all hits for GT check
    result['rag_n_candidates']  = len(hits)
    result['rag_query']         = build_retrieval_query(query_rows, events_m)
    result['latency_s']         = round(elapsed, 2)
    result['top_ae_score']      = float(np.max(event_scores))
    result['event_summary']     = event_summary
    result['sigma_eids_used']   = sigma_eids_used
    result['raw_response']      = raw
    result['prompt']           = prompt
    return result


print('Attribution utilities ready.')
print(f'RAG k={RAG_K}  (behavioural summary query)')

Attribution utilities ready.
RAG k=15  (behavioural summary query)


## 4. End-to-end demo

Run the full two-model pipeline on the highest-scoring flagged chain from OTRF Atomic.
The single-event AE selects the 32 most anomalous events from the chain before passing
them to Phi-4.

In [5]:
best = max(flagged['otrf_at'], key=lambda w: w['peak_score'])
gt   = get_chain_gt(best['chain'], events_m)

print(f'Chain length   : {len(best["chain"])} events')
print(f'Seq AE score   : {best["peak_score"]:.4f}  (threshold={SEQ_THRESHOLD})')
print(f'GT technique   : {gt}')
print()

# Show single-event AE score distribution before selection
all_rows = list(best['chain'])
all_scores = score_events(ae_model, all_rows, X_m_ae, DEVICE)
print(f'Single-event AE scores — min={all_scores.min():.4f}  '
      f'mean={all_scores.mean():.4f}  max={all_scores.max():.4f}')

peak_rows, _ = select_peak_events(best['chain'], ae_model, X_m_ae, DEVICE)
print(f'Top-{TOP_N_EVENTS} selected rows: {peak_rows[0]}..{peak_rows[-1]}')
print()

event_text = format_events_for_slm(peak_rows, events_m)
print('=== Formatted events ===')
print(event_text)
print()

result = attribute_chain(best, events_m)
print('=== RAG retrieval query ===')
print(result['rag_query'])
print()
print('=== Full prompt sent to LLM ===')
print(result['prompt'])
print()
print('=== Attribution result ===')
print(f'Technique    : [{result["technique_id"]}] {result["technique_name"]}')
print(f'Confidence   : {result["confidence"]}')
print(f'RAG top-{RAG_K}  : {result["rag_candidates"]}')
print(f'GT in RAG    : {gt in result["rag_candidates"]}')
print(f'Latency      : {result["latency_s"]}s')
print()
print('Explanation:')
print(result.get('explanation', ''))
print()
print('Recommended action:')
print(result.get('recommended_action', ''))
print()
gt_stripped  = gt.strip()
exact_match  = result['technique_id'] == gt_stripped
tactic_match = result['technique_id'].split('.')[0] == gt_stripped.split('.')[0] if gt_stripped else None
in_rag       = gt_stripped in result['rag_candidates']
print(f'GT technique : {repr(gt_stripped)}')
print(f'Exact match  : {exact_match}')
print(f'Tactic match : {tactic_match}')
print(f'GT in RAG    : {in_rag}  (candidates: {result["rag_candidates"]})')

Chain length   : 45 events
Seq AE score   : 2.7528  (threshold=0.627528)
GT technique   : T1021.003

Single-event AE scores — min=1.7605  mean=2.8603  max=5.3295
Top-32 selected rows: 842643..881083

=== Formatted events ===
Event 1: EID=13 (Registry Value Set)
  image: C:\WindowsAzure\GuestAgent_2.7.41491.993_2020-09-17_150921\GuestAgent\WindowsAzureGuestAgent.exe
  target_object: HKLM\SOFTWARE\Microsoft\Windows Azure\HandlerState\Microsoft.Azure.NetworkWatcher.NetworkWatcherAgentWindows_1.4.1654.1\InstallState
  details: Enabled
---
Event 2: EID=13 (Registry Value Set)
  image: C:\WindowsAzure\GuestAgent_2.7.41491.993_2020-09-17_150921\GuestAgent\WindowsAzureGuestAgent.exe
  target_object: HKLM\SOFTWARE\Microsoft\Windows Azure\HandlerState\Microsoft.Azure.NetworkWatcher.NetworkWatcherAgentWindows_1.4.1654.1\GuestAgentCode
---
Event 3: EID=13 (Registry Value Set)
  image: C:\WindowsAzure\GuestAgent_2.7.41491.993_2020-09-17_150921\GuestAgent\WindowsAzureGuestAgent.exe
  target_object: 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== RAG retrieval query ===
Suspicious Windows Sysmon activity — event types: Registry Object Create/Delete x38, Registry Value Set x6, DNS Query x1. Key processes: C:\WindowsAzure\GuestAgent_2.7.41491.993_2020-09-17_150921\GuestAgent\WindowsAzureGuestAgent.exe Targets: HKLM\SOFTWARE\Microsoft\Windows Azure\HandlerState\Microsoft.Azure.NetworkWatcher.NetworkWatcherAgentWindows_1.4.1654.1\InstallState, HKLM\SOFTWARE\Microsoft\Windows Azure\HandlerState\Microsoft.Azure.NetworkWatcher.NetworkWatcherAgentWindows_1.4.1654.1\GuestAgentMessage, HKLM\SOFTWARE\Microsoft\Windows Azure\HandlerState\Microsoft.Azure.NetworkWatcher.NetworkWatcherAgentWindows_1.4.1654.1\GuestAgentCode, HKLM\System\CurrentControlSet\Services\Tcpip\Parameters DNS queries: md-xgq5px1vx0km.blob.core.windows.net

=== Full prompt sent to LLM ===
You are a cybersecurity analyst reviewing Windows Sysmon events flagged as anomalous.

## Events in Flagged Chain Window
Event 1: EID=13 (Registry Value Set)
  image: C:\WindowsAzu

## 5. Accuracy evaluation

Sample `N_EVAL` flagged chains per source, run the full two-model attribution pipeline,
and compare predictions against ground-truth T-codes.

**Metrics**
- **Exact match** — predicted T-code equals the GT T-code
- **Tactic-level match** — same parent technique (e.g. T1003 vs T1003.001)
- **GT in RAG top-k** — whether the correct technique was retrieved as a candidate
  (the RAG recall ceiling — if this is low, retrieval is the bottleneck)

In [6]:
random.seed(42)
eval_results = []
SEP = '=' * 72

for source in ['otrf_at', 'splunk']:
    chains_with_gt = [
        (entry, get_chain_gt(entry['chain'], events_m))
        for entry in flagged[source]
    ]
    chains_with_gt = [(entry, gt.strip()) for entry, gt in chains_with_gt if gt.strip()]

    sample = random.sample(chains_with_gt, min(N_EVAL, len(chains_with_gt)))
    print(f'\n{SEP}')
    print(f'=== {source}  ({len(sample)} chains)')
    print(SEP)

    for entry, gt in sample:
        try:
            # Pre-check RAG: if GT not retrievable the model cannot be correct.
            # That is a RAG failure; run attribution only on RAG-hit chains.
            _all_rows = list(entry['chain'])
            _qrows    = _all_rows if len(_all_rows) <= 300 else sorted(random.sample(_all_rows, 300))
            _rag_hits = retrieve_multi_query(_qrows, events_m, k=RAG_K)
            _rag_tids = [h['technique_id'] for h in _rag_hits]
            _in_rag   = gt in _rag_tids

            if not _in_rag:
                print(f'  [RAG-miss]  GT={gt:<14} not in {len(_rag_tids)} candidates — skipping model')
                eval_results.append({
                    'source': source, 'gt': gt, 'gt_parent': gt.split('.')[0],
                    'pred': '', 'pred_parent': '',
                    'exact': False, 'parent_match': False,
                    'gt_in_rag': False, 'model_ran': False,
                    'confidence': '', 'explanation': '', 'recommended_action': '',
                    'latency_s': 0, 'top_ae_score': 0,
                    'event_summary': '', 'rag_n_candidates': len(_rag_tids),
                    'sigma_eids_used': [],
                })
                continue

            res          = attribute_chain(entry, events_m)
            pred         = res['technique_id']
            gt_parent    = gt.split('.')[0]
            pred_parent  = pred.split('.')[0]
            exact        = pred == gt
            parent_match = pred_parent == gt_parent
            in_rag       = gt in res['rag_candidates']
            reasoning    = extract_reasoning(res.get('raw_response', ''))

            eval_results.append({
                'source':            source,
                'gt':                gt,
                'gt_parent':         gt_parent,
                'pred':              pred,
                'pred_parent':       pred_parent,
                'exact':             exact,
                'parent_match':      parent_match,
                'gt_in_rag':         in_rag,
                'model_ran':         True,
                'confidence':        res.get('confidence', ''),
                'explanation':       res.get('explanation', ''),
                'recommended_action': res.get('recommended_action', ''),
                'latency_s':         res.get('latency_s', 0),
                'top_ae_score':      res.get('top_ae_score', 0),
                'event_summary':     res.get('event_summary', ''),
                'rag_n_candidates':  res.get('rag_n_candidates', 0),
                'sigma_eids_used':   res.get('sigma_eids_used', []),
            })

            marker   = 'v' if exact else ('~' if parent_match else 'x')
            rag_tag  = f'rag=HIT({res.get("rag_n_candidates",0)})'
            print(f'  [{marker}]  GT={gt:<14} pred={pred:<14} '
                  f'parent={"HIT" if parent_match else "mis"}  '
                  f'{rag_tag:<14} conf={res.get("confidence","?"):<7} {res["latency_s"]:.0f}s')
            sigma_eids = res.get('sigma_eids_used', [])
            sigma_tag  = f'  [Sigma EIDs: {sigma_eids}]' if sigma_eids else ''
            print(f'       Events   : {res.get("event_summary", "(none)")}{sigma_tag}')
            if reasoning:
                print(f'       Reasoning: {reasoning}')
            print()
        except Exception as exc:
            print(f'  ERROR: {exc}')

df_eval = pd.DataFrame(eval_results)
n = len(df_eval)
print(f'\nEvaluation complete: {n} results.')
if n:
    df_ran = df_eval[df_eval['model_ran']] if 'model_ran' in df_eval.columns else df_eval
    print()
    for _src in df_eval['source'].unique():
        sub     = df_eval[df_eval['source'] == _src]
        sub_ran = sub[sub['model_ran']] if 'model_ran' in sub.columns else sub
        print(f'{_src}  (n={len(sub)},  model ran on {len(sub_ran)} GT-in-RAG chains)')
        print(f'  RAG recall        : {sub["gt_in_rag"].mean():.1%}  (avg {sub["rag_n_candidates"].mean():.0f} candidates)')
        if len(sub_ran):
            print(f'  Model exact match : {sub_ran["exact"].mean():.1%}  (conditional on GT in RAG)')
            print(f'  Model parent match: {sub_ran["parent_match"].mean():.1%}')
            print(f'  Avg latency       : {sub_ran["latency_s"].mean():.0f}s')
        print()
    print('-' * 40)
    n_ran = len(df_ran)
    print(f'Overall  (n={n},  model ran on {n_ran} GT-in-RAG chains)')
    print(f'  RAG recall        : {df_eval["gt_in_rag"].mean():.1%}')
    if n_ran:
        print(f'  Model exact match : {df_ran["exact"].mean():.1%}  (conditional on GT in RAG)')
        print(f'  Model parent match: {df_ran["parent_match"].mean():.1%}')
        print(f'  Pipeline exact    : {df_eval["exact"].mean():.1%}  (all chains incl. RAG misses)')
        print(f'  Avg latency       : {df_ran["latency_s"].mean():.0f}s')


=== otrf_at  (20 chains)
  [RAG-miss]  GT=T1021          not in 14 candidates — skipping model
  [RAG-miss]  GT=T1021.003      not in 22 candidates — skipping model
  [x]  GT=T1003.002      pred=T1055          parent=mis  rag=HIT(29)    conf=medium  92s
       Events   : EID=12(Registry Object Create/Delete) x22, EID=13(Registry Value Set) x16, EID=9(?) x6, EID=11(File Create) x1  [Sigma EIDs: [1, 12, 13]]
       Reasoning: - Process names: `reg.exe`, `wmic.exe`, `lsass.exe`- Registry paths: `HKLM\SYSTEM\CurrentControlSet\Control\Lsa`, `SOFTWARE\Microsoft\Windows\CurrentVersion\Run`- Command lines involving registry modifications and process creation- File access to sensitive directories like `System32` and `Sysvol`

  [x]  GT=T1547.001      pred=T1562.001      parent=mis  rag=HIT(22)    conf=high    88s
       Events   : EID=13(Registry Value Set) x24, EID=7(Image Load) x15, EID=12(Registry Object Create/Delete) x5, EID=1(Process Create) x1  [Sigma EIDs: [1, 7, 10, 11, 13]]
       Re